# 02 — How far is the released selector from the global optimum?

Use the real Qwen activation archive, the fitted SSD cost from Notebook 01, and the released vlm-flash selector. For each greedy mask, solve the global fixed-cardinality problem at the **same realized row count**.

\[
\eta_R^\star=\max_{\|M\|_1=R}\frac{I(M)}{L(M)}.
\]

The four methods are Global DP, upstream greedy, saturation tiles, and Top-$R$. The tile rule is restored from [the reference notebook](https://github.com/yabsed/vlm-in-flash/blob/f3620ce022a7b7bcb69a4bbb7be043343d90910f/conclusion/python/results/02/02_global_oracle.ipynb).

In [ ]:
from pathlib import Path
from itertools import product
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter1d
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".gitmodules").is_file())
MODEL = ROOT / "conclusion/python/results/01/latency_model.json"
TRACE = ROOT / "experiments/22_predicted_lambda_trim/results/activation_traces.npz"
UPSTREAM = ROOT / "conclusion/upstream/vlm-flash"
RESULTS = Path("oracle_results_tiles.csv")

RUN_ORACLE = False
MAX_TRACES_PER_SHAPE = 32       # 0 = all 504 traces
BUDGETS = tuple(np.arange(1, 10) / 10)
RTOL = 1e-6
PAPER_IMPL = "native"

## One measured model serves every projection.

Notebook 01 gives

\[
T_{\rm KiB}(z)=c_1z+\delta\max(s_{\rm KiB},z).
\]

If one weight row occupies \(q\) KiB, then

\[
T_{\rm row}(r)=b_1r+(b_2-b_1)\max(s,r),
\qquad
s=s_{\rm KiB}/q,\quad b_1=qc_1,\quad b_2=q(c_1+\delta).
\]

In [ ]:
model = json.loads(MODEL.read_text())
s_kib = model["s_kib"]
c1 = model["c1_ms_per_kib"]
delta = model["delta_ms_per_kib"]
s99 = model["s99_kib"]

display(Markdown(f"$s={s_kib:g}$ KiB, $c_1={1000*c1:.6f}$ µs/KiB, $\\delta={1000*delta:.6f}$ µs/KiB."))

In [ ]:
def costs(n, s, b1, b2):
    r = np.arange(n + 1, dtype=float)
    T = b1 * r + (b2 - b1) * np.maximum(s, r)
    T[0] = 0
    return T


def runs(mask):
    return np.flatnonzero(np.diff(np.r_[False, mask, False])).reshape(-1, 2)


def metrics(v, mask, T):
    length = np.diff(runs(mask), axis=1).ravel()
    return v[mask].sum(), T[length].sum()


def top_r(v, R):
    mask = np.zeros(len(v), bool)
    mask[np.argsort(-v, kind="stable")[:R]] = True
    return mask

## The last zero makes the ratio problem additive.

For a proposed ratio \(\lambda\),

\[
F_R(\lambda)=\max_{\|M\|_1=R}\{I(M)-\lambda L(M)\},
\qquad
F_R(\lambda)\ge0\iff\lambda\le\eta_R^\star.
\]

Let \(D_{z,r}\) be the best additive value among prefixes containing \(z\) zeros and \(r\) ones. Splitting at the last zero gives

\[
D_{z,r}=
\max_{0\le t\le r}
[D_{z-1,t}+P_{z+r}-P_{z+t}-\lambda T(r-t)].
\]

The two-line form turns the scan over \(t\) into one moving maximum for short runs and one prefix maximum for long runs.

In [ ]:
def bellman(v, R, lam, s, b1, b2, keep=False):
    Z = len(v) - R
    P = np.r_[0, np.cumsum(v)]
    r = np.arange(R + 1)
    prev = P[:R + 1] - lam * costs(R, s, b1, b2)
    table = np.empty((Z + 1, R + 1)) if keep else None
    if keep:
        table[0] = prev

    h = min(int(np.floor(s)), R)
    for z in range(1, Z + 1):
        prefix = P[z:z + R + 1]
        base = prev - prefix

        short = np.full(R + 1, -np.inf)
        if h:
            short[1:] = maximum_filter1d(
                base[:-1] + lam * b1 * r[:-1],
                size=h, origin=(h - 1) // 2, mode="constant", cval=-np.inf)

        long = np.full(R + 1, -np.inf)
        long[h + 1:] = np.maximum.accumulate(base + lam * b2 * r)[:R - h]

        prev = np.maximum.reduce([
            prev,
            prefix - lam * ((b2 - b1) * s + b1 * r) + short,
            prefix - lam * b2 * r + long,
        ])
        if keep:
            table[z] = prev

    return table if keep else prev[-1]

In [ ]:
def recover(v, R, lam, T, table):
    P = np.r_[0, np.cumsum(v)]
    mask = np.zeros(len(v), bool)
    r = R
    for z in range(len(v) - R, 0, -1):
        t = np.arange(r + 1)
        score = table[z - 1, :r + 1] + P[z + r] - P[z + t] - lam * T[r - t]
        t = int(np.argmax(score))
        mask[z + t:z + r] = True
        r = t
    mask[:r] = True
    return mask

In [ ]:
def solve(v, R, s, b1, b2):
    T = costs(len(v), s, b1, b2)
    initial = top_r(v, R)
    I, L = metrics(v, initial, T)
    if I == 0:
        return initial, 0

    lo, hi = I / L, I / (b2 * R)
    while hi - lo > RTOL * lo:
        lam = (lo + hi) / 2
        if bellman(v, R, lam, s, b1, b2) >= 0:
            lo = lam
        else:
            hi = lam

    table = bellman(v, R, lo, s, b1, b2, keep=True)
    mask = recover(v, R, lo, T, table)
    I, L = metrics(v, mask, T)
    return mask, I / L

## Saturation tiles use the same boundary as the cost model.

In the reference notebook, the cost boundary and the tile width both used $s$.
Here the fitted boundary comes from Notebook 01, so an integer tile contains

$$B=\left\lceil\frac{s_{\rm KiB}}{q}\right\rceil\quad\text{rows}.$$

The table below displays both $B$ and $qB$. No 240 KiB constant is inserted.
This is the **fitted** constant-cost-per-byte boundary, not a second estimate of
physical saturation. The separate $s_{99}$ still limits upstream candidates.

For each offset $0$ and $\lfloor B/2\rfloor$, select the $\lfloor R/B\rfloor$
most important full tiles, then the best disjoint interval of length $R\bmod B$.
Keep the mask with the better merged-run ratio. If $R<B$, choose the best
length-$R$ interval. This also covers a tile wider than the entire projection.
The function below is unchanged from the reference; its argument `s` is the integer $B$.

In [ ]:
def tiles(v, R, T, s):
    P = np.r_[0., np.cumsum(v)]
    if R < s:
        start = np.argmax(P[R:] - P[:-R])
        mask = np.zeros(len(v), bool)
        mask[start:start + R] = True
        return mask
    choices = []
    k, remaining = divmod(R, s)
    for offset in (0, s // 2):
        starts = np.arange(offset, len(v) - s + 1, s)
        if len(starts) < k:
            continue
        weights = P[starts + s] - P[starts]
        mask = np.zeros(len(v), bool)
        for start in starts[np.argsort(-weights, kind="stable")[:k]]:
            mask[start:start + s] = True
        if remaining:
            occupied = np.r_[0, np.cumsum(mask)]
            valid = occupied[remaining:] == occupied[:-remaining]
            if not valid.any():
                continue
            scores = np.where(valid, P[remaining:] - P[:-remaining], -np.inf)
            start = np.argmax(scores)
            mask[start:start + remaining] = True
        I, L = metrics(v, mask, T)
        choices.append((I / L, mask))
    return max(choices, key=lambda row: row[0])[1]

### A small exhaustive check is enough.

This is not another experiment. It only checks that the accelerated Bellman value equals brute force for small masks, including a fractional boundary.

In [ ]:
v = np.random.default_rng(0).uniform(0, 4, 7)
masks = np.array(list(product([False, True], repeat=len(v))))
checks = 0

for s in (0.5, 2.5, 8):
    T = costs(len(v), s, 0.25, 1.0)
    for R in (1, 3, 7):
        feasible = masks[masks.sum(axis=1) == R]
        for lam in (0.1, 1.0, 10.0):
            exact = max(metrics(v, m, T)[0] - lam * metrics(v, m, T)[1] for m in feasible)
            np.testing.assert_allclose(bellman(v, R, lam, s, 0.25, 1.0), exact)
            mask = tiles(v, R, T, int(np.ceil(s)))
            assert mask.sum() == R
            checks += 1

display(Markdown(f"**{checks}** exhaustive Bellman checks and tile-cardinality checks passed."))

## The experiment uses real dense-model activations.

The Experiment 22 archive stores \(v_i=\operatorname{mean}|x_i|\) for real Qwen projection calls. A cap of 32 gives 128 traces; a cap of 0 uses all 504.

In [ ]:
def load_traces(path, cap):
    with np.load(path, allow_pickle=False) as archive:
        manifest = json.loads(str(archive["manifest_json"].item()))
        traces = []
        for entry in manifest["traces"]:
            trace = {k: v for k, v in entry.items() if k != "array"}
            trace["values"] = np.asarray(archive[entry["array"]], dtype=np.float32)
            traces.append(trace)

    if cap:
        selected = []
        for shape in sorted({t["shape"] for t in traces}):
            group = [t for t in traces if t["shape"] == shape]
            index = np.linspace(0, len(group) - 1, min(cap, len(group)), dtype=int)
            selected += [group[i] for i in index]
        traces = selected

    return sorted(traces, key=lambda t: t["trace_id"]), manifest


traces, manifest = load_traces(TRACE, MAX_TRACES_PER_SHAPE)
weight_bytes = {"float16": 2, "bfloat16": 2, "float32": 4}[manifest["model_dtype"]]

summary = pd.DataFrame([{"shape": t["shape"], "n": t["n"], "d": t["d"]} for t in traces])
summary = summary.groupby(["shape", "n", "d"]).size().rename("traces").reset_index()
summary["KiB/row"] = weight_bytes * summary.d / 1024
summary["boundary (rows)"] = s_kib / summary["KiB/row"]
summary["tile rows"] = np.ceil(summary["boundary (rows)"]).astype(int)
summary["tile KiB"] = summary["tile rows"] * summary["KiB/row"]
display(summary.round(3))
display(Markdown(f"**{len(traces) * len(BUDGETS)}** budget cases."))

## Upstream greedy determines the cardinality.

Run the released selector first. If it realizes \(R_{\rm eff}\) rows, run Global DP, saturation tiles, and Top-\(R\) at exactly \(R_{\rm eff}\).

In [ ]:
PAPER_GEOMETRY = {
    "896x128": (8, 8), "896x896": (8, 8),
    "896x4864": (8, 8), "4864x896": (12, 16),
}


def load_upstream():
    sys.path.insert(0, str((UPSTREAM / "src").resolve()))
    import torch
    from vlmflash import ChunkParams, LatencyTable, select_chunks
    table = LatencyTable({
        k: c1 * k + delta * max(s_kib, k)
        for k in range(1, int(np.ceil(s99)) + 2)
    })
    return torch, ChunkParams, select_chunks, table


def paper_mask(trace, v, R, upstream):
    torch, ChunkParams, select_chunks, table = upstream
    start, jump = PAPER_GEOMETRY[trace["shape"]]
    params = ChunkParams(start_kb=start, end_kb=s99 + 1,
                         step_kb=start, jump_cap_kb=jump)
    row_kib = weight_bytes * trace["d"] / 1024
    selection = select_chunks(
        torch.tensor(v, dtype=torch.float32), R, row_kib,
        table, params, impl=PAPER_IMPL)
    return selection.mask.cpu().numpy().astype(bool)

## Four masks answer the comparison.

Global DP, upstream greedy, saturation tiles, and Top-$R$ use the same realized
cardinality and merged-run fitted cost. The normalized float32 importance
supplied to upstream is also used by the oracle. Empty greedy masks and
zero-importance cases are omitted.

Set `RUN_ORACLE = True` to compute this comparison into `oracle_results_tiles.csv`.
Ordinary evaluation reads that file if present. The earlier three-method
`oracle_results.csv` is left intact; it is not a four-method result.
A new run recomputes the sweep. There is no automatic resume or cache matching.

In [ ]:
def one_case(trace, fraction, upstream):
    v = trace["values"].astype(float)
    v = v / v.mean() if v.sum() else v
    v = v.astype(np.float32).astype(float)

    nominal = max(1, int(fraction * len(v)))
    paper = paper_mask(trace, v, nominal, upstream)
    R = int(paper.sum())
    if R == 0 or v.sum() == 0:
        return []

    row_kib = weight_bytes * trace["d"] / 1024
    s, b1, b2 = s_kib / row_kib, row_kib * c1, row_kib * (c1 + delta)
    T = costs(len(v), s, b1, b2)
    oracle, eta = solve(v, R, s, b1, b2)

    rows = []
    for name, mask in {
        "Global DP": oracle,
        "Upstream greedy": paper,
        "Saturation tiles": tiles(v, R, T, int(np.ceil(s))),
        "Top-R": top_r(v, R),
    }.items():
        assert mask.sum() == R
        I, L = metrics(v, mask, T)
        rows.append({
            "trace_id": trace["trace_id"], "shape": trace["shape"],
            "prompt": trace["input_index"], "fraction": fraction,
            "R_nominal": nominal, "R": R, "method": name,
            "retained_pct": 100 * I / v.sum(), "fitted_ms": L,
            "gap_pct": max(0, 100 * (1 - I / L / eta)),
            "chunks": len(runs(mask)),
        })
    return rows

In [ ]:
if RUN_ORACLE:
    upstream = load_upstream()
    rows = []
    for i, trace in enumerate(traces, 1):
        for fraction in BUDGETS:
            rows += one_case(trace, fraction, upstream)
        pd.DataFrame(rows).to_csv(RESULTS, index=False)
        if i % 10 == 0 or i == len(traces):
            print(f"{i}/{len(traces)} traces")

frame = pd.read_csv(RESULTS) if RESULTS.exists() else pd.DataFrame()
display(Markdown(f"**{len(frame) // 4 if len(frame) else 0}/{len(traces) * len(BUDGETS)}** cases available."))

## The main figure follows the four methods across row budgets.

For each shape, use the lowest available trace ID, not the trace with the most
favorable result. Each point is an actual mask on that one trace, with
$x=L(M)$ and $y=100I(M)/I_{\rm tot}$. Connect points in increasing nominal
retained fraction, $0.1,\ldots,0.9$. Labels on the oracle curve give the realized
$R$ shared by all four methods at that budget.

The logarithmic latency axis follows the reference figure. These are fixed-$R$
trade-off trajectories, **not a complete Pareto frontier**. A ratio optimum
need not dominate each competitor separately in importance and latency.
Shape-level gap summaries below use all available traces; the main figure
does not combine coordinate-wise medians into a fictitious mask.

In [ ]:
if not frame.empty:
    for shape, group in frame.groupby("shape", sort=False):
        trace_id = group.trace_id.min()
        example = group[group.trace_id == trace_id]
        plt.figure(figsize=(8, 4))
        for name, points in example.groupby("method", sort=False):
            points = points.sort_values("fraction")
            plt.plot(points.fitted_ms, points.retained_pct, "o-", label=name, ms=4)
            if name == "Global DP":
                for row in points.itertuples():
                    plt.annotate(str(row.R), (row.fitted_ms, row.retained_pct),
                                 xytext=(3, 5), textcoords="offset points", fontsize=7)
        plt.xscale("log")
        plt.xlabel("Fitted additive latency (ms; log scale)")
        plt.ylabel("Retained importance (%)")
        plt.title(f"{shape}: trace {trace_id}; labels are realized R")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

In [ ]:
if not frame.empty:
    gaps = frame[frame.method != "Global DP"].groupby(
        ["shape", "method"]).gap_pct.median().unstack()
    ax = gaps.plot.bar(figsize=(9, 3.5), rot=0)
    ax.set(xlabel="Projection shape", ylabel="Median gap to global optimum (%)")
    plt.tight_layout()
    plt.show()

In [ ]:
if not frame.empty:
    paper = frame[frame.method == "Upstream greedy"]
    curves = paper.groupby(["fraction", "shape"]).gap_pct.median().unstack()
    ax = curves.plot(figsize=(8, 3.5), marker=".")
    ax.set(xlabel="Nominal retained fraction", ylabel="Median greedy gap (%)")
    plt.tight_layout()
    plt.show()

## Scope

Global DP solves the fitted two-line additive fixed-cardinality objective to
numerical tolerance $10^{-6}$. The displayed gap is relative to its recovered
ratio, not an arbitrary-lookup optimum. The fitted curve is extrapolated for
runs outside Notebook 01's measured range; native task splitting and scheduling
are not modeled here. This is neither end-to-end quality nor measured
complete-mask SSD wall time.

The real inputs remain the archived dense Qwen activation summaries and the
measured coefficients from Notebook 01. Traces from three prompts do not
constitute thousands of independent prompts. Hashes, resume infrastructure,
and a separate scaling experiment are not needed for this comparison.